# OpsLM-v2 — DPO preference alignment (Colab T4)

Direct Preference Optimization on top of the SFT model `dhf1234/OpsLM-v1`.
Teaches OpsLM to prefer grounded, hedged answers over confident hallucinations.
Thin driver over `training/scripts/train_opslm_dpo.py`. Needs a HF write token.

Runtime -> Change runtime type -> **T4 GPU**. ~1-2h (DPO is 1 epoch, low LR).

## 1. Names + token

In [ ]:
import os
from getpass import getpass

HF_USER = "dhf1234"                    # your HF username
BASE = f"{HF_USER}/OpsLM-v1"           # the SFT model to align
PUSH_REPO = f"{HF_USER}/OpsLM-v2"      # the DPO output
os.environ["HF_TOKEN"] = getpass("HF token (write access): ")
print("aligning", BASE, "->", PUSH_REPO)

## 2. Install pinned deps

In [ ]:
!pip -q install "unsloth==2025.*" "trl>=0.12,<0.20" "datasets>=3" "huggingface_hub>=0.25"

## 3. Get the code + the DPO preference data

The preference set is built locally with
`python -m opsverse_training.generate_preferences` and committed to
`data/dpo/{train,val}.jsonl` (TRL conversational format). The clone has it.

In [ ]:
!git clone --depth 1 https://github.com/dilipna/OPsVerse.git
%cd OPsVerse

import os
assert os.path.exists("data/dpo/train.jsonl"), "generate data/dpo first (see training/README.md)"
print(sum(1 for _ in open("data/dpo/train.jsonl")), "preference pairs")

## 4. DPO (resumable)

Re-run with `--resume` appended if the session drops; checkpoints push to Hub.

In [ ]:
!python training/scripts/train_opslm_dpo.py \n    --base-model {BASE} \n    --data-repo data/dpo \n    --push-repo {PUSH_REPO} \n    --epochs 1 --beta 0.1 --save-steps 50

## 5. After: before/after eval

Serve v1 and v2, run the Phase-4 eval + the structured-output eval against
each, and write the delta into `docs/reports/`. DPO should hold or improve
faithfulness/citation-use without breaking JSON fidelity.